# 01 — Data Exploration

Load PTB-XL and MIT-BIH datasets, inspect shapes, class distributions, signal statistics, and lead correlations.

In [ ]:
import sys, os
sys.path.insert(0, os.path.join('..', 'src'))
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.express as px

import config
from data_loader import load_ptbxl, load_mitbih, ECGDataset

%matplotlib inline
plt.rcParams['figure.dpi'] = 120
sns.set_theme(style='whitegrid')

## 1. Load PTB-XL

In [ ]:
# Download command (run once):
# import wfdb; wfdb.dl_database('ptb-xl', str(config.PATHS.ptbxl))

try:
    ptbxl = load_ptbxl(config.PATHS.ptbxl, sampling_rate=100)
    print(ptbxl)
    print(f'X shape : {ptbxl.X.shape}')
    print(f'y shape : {ptbxl.y.shape}')
    print(f'Labels  : {ptbxl.labels}')
except FileNotFoundError as e:
    print(f'[INFO] PTB-XL not downloaded yet:\n  {e}')
    ptbxl = None

## 2. Load MIT-BIH

In [ ]:
# Download command (run once):
# import wfdb; wfdb.dl_database('mitdb', str(config.PATHS.mitbih))

try:
    mitbih = load_mitbih(config.PATHS.mitbih)
    print(mitbih)
    print(f'X shape : {mitbih.X.shape}')
    print(f'y shape : {mitbih.y.shape}')
    print(f'Labels  : {mitbih.labels}')
except FileNotFoundError as e:
    print(f'[INFO] MIT-BIH not downloaded yet:\n  {e}')
    mitbih = None

## 3. Class imbalance bar charts

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

if ptbxl is not None:
    counts = ptbxl.y.sum(axis=0)
    axes[0].bar(ptbxl.labels, counts, color='steelblue')
    axes[0].set_title('PTB-XL Superclass Distribution')
    axes[0].set_ylabel('Count')

if mitbih is not None:
    unique, cnt = np.unique(mitbih.y, return_counts=True)
    axes[1].bar([mitbih.labels[u] for u in unique], cnt, color='darkorange')
    axes[1].set_title('MIT-BIH AAMI Class Distribution')
    axes[1].set_ylabel('Count')

plt.tight_layout()
plt.show()

## 4. Plot 5 random ECG samples per class (PTB-XL)

In [ ]:
if ptbxl is not None:
    LEAD_NAMES = ['I','II','III','aVR','aVL','aVF','V1','V2','V3','V4','V5','V6']
    rng = np.random.default_rng(42)
    n_show = 5

    for cls_idx, cls_name in enumerate(ptbxl.labels):
        mask = ptbxl.y[:, cls_idx] == 1
        cls_X = ptbxl.X[mask]
        if len(cls_X) == 0:
            continue
        sel = rng.choice(len(cls_X), size=min(n_show, len(cls_X)), replace=False)

        fig, axes = plt.subplots(len(LEAD_NAMES), min(n_show, len(sel)),
                                 figsize=(min(n_show, len(sel)) * 3, len(LEAD_NAMES) * 0.8),
                                 sharex=True)
        if axes.ndim == 1:
            axes = axes[:, np.newaxis]

        fig.suptitle(f'Class: {cls_name}  ({mask.sum()} total samples)', fontsize=10)

        for col_i, s_idx in enumerate(sel):
            for lead_i, lead_name in enumerate(LEAD_NAMES):
                axes[lead_i, col_i].plot(cls_X[s_idx, lead_i], lw=0.6)
                axes[lead_i, col_i].set_ylabel(lead_name, fontsize=6)
                axes[lead_i, col_i].tick_params(labelsize=5)

        plt.tight_layout()
        plt.show()
else:
    print('PTB-XL not loaded — skipping plot.')

## 5. Signal statistics per class

In [ ]:
if ptbxl is not None:
    rows = []
    for cls_idx, cls_name in enumerate(ptbxl.labels):
        mask = ptbxl.y[:, cls_idx] == 1
        cls_X = ptbxl.X[mask]
        if len(cls_X) == 0:
            continue
        rows.append({
            'class': cls_name,
            'n_samples': int(mask.sum()),
            'duration_s': round(cls_X.shape[-1] / ptbxl.fs, 2),
            'signal_mean': round(float(cls_X.mean()), 4),
            'signal_std': round(float(cls_X.std()), 4),
            'signal_min': round(float(cls_X.min()), 4),
            'signal_max': round(float(cls_X.max()), 4),
        })
    stats_df = pd.DataFrame(rows)
    display(stats_df)
else:
    print('PTB-XL not loaded.')

## 6. Lead correlation heatmap (mean over samples)

In [ ]:
if ptbxl is not None:
    # Average signal across time per sample → (n_samples, n_leads)
    lead_means = ptbxl.X.mean(axis=2)  # (n_samples, 12)
    corr = np.corrcoef(lead_means.T)   # (12, 12)

    LEAD_NAMES = ['I','II','III','aVR','aVL','aVF','V1','V2','V3','V4','V5','V6']
    fig, ax = plt.subplots(figsize=(8, 7))
    sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdBu', center=0,
                xticklabels=LEAD_NAMES, yticklabels=LEAD_NAMES, ax=ax)
    ax.set_title('Lead Correlation Heatmap (PTB-XL)')
    plt.tight_layout()
    plt.show()
else:
    print('PTB-XL not loaded.')